In [2]:
import pandas as pd
import re
from pathlib import Path

# =============================
# Input files
# =============================
SVD_FILE = "/home/moshtasa/Research/phd-svd-recsys/Book/SVD_numbers.txt"
KNN_FILE = "/home/moshtasa/Research/phd-svd-recsys/Book/KNN_numbers.txt"

OUTPUT_DIR = "/home/moshtasa/Research/phd-svd-recsys/Book/csv_outputs"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

TOP_K_LIST = [15, 25, 35]

# =============================
# Parser
# =============================
def parse_file(file_path):
    """
    Returns dict:
    data[topk][genre][n] = avg_user
    """
    data = {}

    with open(file_path, "r") as f:
        for line in f:
            match = re.search(
                r"\[(\w+)\]\s*\[(\d+)\].*?avg/user=([\d.]+)", line
            )
            if not match:
                continue

            genre = match.group(1)
            topk = int(match.group(2))
            avg_user = float(match.group(3))

            if topk not in TOP_K_LIST:
                continue

            # synthetic n or ORIGINAL
            n_match = re.search(r"_(\d+)u_", line)
            n = int(n_match.group(1)) if n_match else "original"

            data.setdefault(topk, {}).setdefault(genre, {})[n] = avg_user

    return data

# =============================
# Parse both datasets
# =============================
svd_data = parse_file(SVD_FILE)
knn_data = parse_file(KNN_FILE)

# =============================
# Collect global genres & n
# =============================
genres = set()
n_values = set()

for dataset in [svd_data, knn_data]:
    for topk in dataset:
        for genre in dataset[topk]:
            genres.add(genre)
            n_values.update(dataset[topk][genre].keys())

genres = sorted(genres)
n_values = sorted(
    n_values,
    key=lambda x: float("inf") if x == "original" else x
)

# =============================
# Build tables (ONE per Top-K)
# =============================
for topk in TOP_K_LIST:
    table = {}

    for n in n_values:
        row = []
        for genre in genres:
            # SVD
            row.append(
                svd_data.get(topk, {}).get(genre, {}).get(n, None)
            )
            # KNN
            row.append(
                knn_data.get(topk, {}).get(genre, {}).get(n, None)
            )
        table[n] = row

    # MultiIndex columns
    columns = pd.MultiIndex.from_tuples(
        [(genre, method) for genre in genres for method in ["SVD", "KNN"]],
        names=["Genre", "Method"]
    )

    df = pd.DataFrame.from_dict(
        table,
        orient="index",
        columns=columns
    )
    df.index.name = "n"

    output_path = f"{OUTPUT_DIR}/TopK_{topk}.csv"
    df.to_csv(output_path)

    print(f"Saved: {output_path}")


Saved: /home/moshtasa/Research/phd-svd-recsys/Book/csv_outputs/TopK_15.csv
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/csv_outputs/TopK_25.csv
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/csv_outputs/TopK_35.csv


###updated original 0102

In [ ]:
#!/usr/bin/env python3

import pandas as pd
from pathlib import Path
import re

# =============================
# Paths
# =============================

CSV_DIR = Path(
    "/home/moshtasa/Research/phd-svd-recsys/Book/csv_outputs"
)

NMF_ORIG_BASE = Path(
    "/home/moshtasa/Research/phd-svd-recsys/Book/NMF/result/rec/top_re/1229_NMF_ORIGINAL/output_0102"
)

TOP_K_LIST = [15, 25, 35]

# =============================
# Helper: read avg.txt
# =============================
def read_avg_file(fp: Path):
    """
    Returns dict:
      {15: avg, 25: avg, 35: avg}
    """
    out = {}
    with open(fp, "r") as f:
        for line in f:
            m = re.match(r"Top-(\d+):\s*([\d.]+)", line.strip())
            if m:
                out[int(m.group(1))] = float(m.group(2))
    return out

# =============================
# Main replacement logic
# =============================

for topk in TOP_K_LIST:
    csv_path = CSV_DIR / f"TopK_{topk}.csv"
    if not csv_path.exists():
        print(f"[skip] Missing {csv_path}")
        continue

    df = pd.read_csv(csv_path, header=[0, 1], index_col=0)

    if "original" not in df.index:
        print(f"[warn] No original row in {csv_path}")
        continue

    # Loop over genres in columns
    genres = sorted(set(g for g, _ in df.columns))

    for genre in genres:
        genre_key = genre.lower().replace(" ", "_").replace("'", "")
        avg_file = (
            NMF_ORIG_BASE
            / f"{genre_key}_explanation"
            / f"{genre_key}__original.avg.txt"
        )

        if not avg_file.exists():
            print(f"[warn] Missing avg file: {avg_file}")
            continue

        avg_values = read_avg_file(avg_file)

        if topk not in avg_values:
            print(f"[warn] Top-{topk} not found in {avg_file}")
            continue

        # 🔴 REPLACE ONLY THIS CELL
        df.loc["original", (genre, "SVD")] = avg_values[topk]

    # Save back (overwrite)
    df.to_csv(csv_path)
    print(f"[OK] Updated SVD original row in {csv_path}")


[OK] Updated NMF original row in /home/moshtasa/Research/phd-svd-recsys/Book/csv_outputs/TopK_15.csv
[OK] Updated NMF original row in /home/moshtasa/Research/phd-svd-recsys/Book/csv_outputs/TopK_25.csv
[OK] Updated NMF original row in /home/moshtasa/Research/phd-svd-recsys/Book/csv_outputs/TopK_35.csv


In [7]:
import pandas as pd
import os

# Folder where CSVs are saved
folder = "/home/moshtasa/Research/phd-svd-recsys/Book/"

# Files for Top 15, 25, 35
files = ["Top15_table.csv", "Top25_table.csv", "Top35_table.csv"]

for file in files:
    path = os.path.join(folder, file)
    if os.path.exists(path):
        df = pd.read_csv(path, header=[0, 1], index_col=0)  # MultiIndex columns, first column = index
        print(f"\n=== {file} ===")
        print(df)
    else:
        print(f"File {file} not found in {folder}")



=== Top15_table.csv ===
Genre    Adult        Adventure        Classics        Drama        Fantasy  \
Method     SVD    NMF       SVD    NMF      SVD    NMF   SVD    NMF     SVD   
n                                                                             
2         0.76   0.17      2.84   3.50     1.35   1.16  5.00   4.74    3.82   
4         2.21   0.78      3.23   2.74     1.88   0.92  5.46   4.62    3.99   
6         2.73   0.74      3.62   3.09     2.21   1.20  5.20   4.90    5.01   
25        8.63   2.47     10.33   4.58     4.57   2.04  9.52   5.91    4.83   
50        9.60   4.56      9.67   6.06     4.24   4.18  8.26   6.53    5.50   
100       9.66   7.61      9.55   8.24     4.25   7.19  8.06   8.64    5.52   
200       8.17  10.63      7.57  11.11     3.45  10.50  6.75  11.52    5.46   
300       6.99  11.74      6.87  12.12     3.51  11.72  6.39  12.48    5.32   
350       4.78  11.66      6.95  11.77     4.06  11.62  6.53  12.28    5.19   
500       4.30  12.41      

In [18]:
#!/usr/bin/env python3
import re
import pandas as pd
from pathlib import Path

# ========= CONFIG =========
INPUT_CSV   = Path("/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/df_final_with_genres.csv")
OUT_DIR     = Path("/home/moshtasa/Research/phd-svd-recsys/Book")
SUMMARY_TXT = OUT_DIR / "summary.txt"
SUMMARY_CSV = OUT_DIR / "summary.csv"

GENRE_COL   = "genres"
USER_COL    = "user_id"
BOOK_COL    = "book_id"
RATING_COL  = "rating"

RUNS = [2 ,4 ,6 ,25 ,50 ,100 ,200 ,300 ,350 , 500 ,1000]

POS_RATING  = 5
NEG_RATING  = 1
BLOCK = 1_000_000

def sanitize_fn(s: str) -> str:
    s = (s or "").strip().replace(" ", "_")
    return re.sub(r"[^0-9A-Za-z_]+", "_", s) or "UNK"

def primary_genre(cell: str) -> str:
    if not isinstance(cell, str) or not cell.strip():
        return ""
    return cell.split(",")[0].strip()

def main():
    # Ensure output directory exists
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Load data
    df = pd.read_csv(INPUT_CSV)
    df[GENRE_COL] = df[GENRE_COL].fillna("").astype(str)
    df[USER_COL] = df[USER_COL].astype(int)
    df[BOOK_COL] = df[BOOK_COL].astype(int)

    base_start_uid = df[USER_COL].max() + 1
    orig_users = df[USER_COL].nunique()
    orig_rows = len(df)

    # Build genre lookup
    book_gen = df[[BOOK_COL, GENRE_COL]].drop_duplicates(subset=[BOOK_COL]).copy()
    book_gen["_primary"] = book_gen[GENRE_COL].apply(primary_genre)
    book_gen = book_gen[book_gen["_primary"] != ""]
    all_books = sorted(book_gen[BOOK_COL].unique())
    book_to_genres = dict(book_gen[[BOOK_COL, GENRE_COL]].values)

    # Group books by primary genre
    per_genre = (
        book_gen.groupby("_primary")[BOOK_COL]
        .apply(lambda x: sorted(x.unique()))
        .reset_index()
        .rename(columns={BOOK_COL: "pos_books"})
    )

    target_genres = sorted(per_genre["_primary"].unique())
    summary_rows = []

    with open(SUMMARY_TXT, "w") as log:
        log.write(f"BASE DATA: {orig_users} users, {orig_rows} rows\n")
        log.write(f"NEG_RATING = {NEG_RATING} (NO SAMPLING)\n\n")

    total_new_users = 0
    total_new_rows = 0

    for gi, genre in enumerate(target_genres):
        pos_books = per_genre.loc[per_genre["_primary"] == genre, "pos_books"].iloc[0]
        neg_books = [b for b in all_books if b not in set(pos_books)]
        safe_name = sanitize_fn(genre)

        log_text = f"Genre: {genre}\n"
        log_text += f"  Positive books: {len(pos_books)}\n"
        log_text += f"  Negative books: {len(neg_books)}\n"
        log_text += f"  Original users: {orig_users}\n"
        log_text += f"  Original rows: {orig_rows}\n"
        log_text += f"  Simulating synthetic users:\n"
        print(log_text)
        with open(SUMMARY_TXT, "a") as log:
            log.write(log_text)

        for r_i, run in enumerate(RUNS):
            # Each new user rates all books: pos + neg
            new_rows = run * (len(pos_books) + len(neg_books))
            summary_rows.append({
                "genre": genre,
                "run_users": run,
                "pos_books": len(pos_books),
                "neg_books": len(neg_books),
                "new_rows": new_rows,
                "expected_output_file": str(OUT_DIR / f"f_{safe_name}_{run}u_pos5_neg1_all.csv")
            })
            total_new_users += run
            total_new_rows += new_rows

            log_text = f"    Run {r_i+1}: {run} new users → {new_rows} new rows\n"
            print(log_text)
            with open(SUMMARY_TXT, "a") as log:
                log.write(log_text)

    # Total summary
    final_summary = f"\nTOTAL new synthetic users: {total_new_users}\nTOTAL new rows: {total_new_rows}\n"
    print(final_summary)
    with open(SUMMARY_TXT, "a") as log:
        log.write(final_summary)

    # Save CSV summary
    pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)
    print(f"✅ Summary saved to CSV: {SUMMARY_CSV} and TXT: {SUMMARY_TXT}")

if __name__ == "__main__":
    main()


Genre: Adult
  Positive books: 106
  Negative books: 9309
  Original users: 53424
  Original rows: 5976479
  Simulating synthetic users:

    Run 1: 2 new users → 18830 new rows

    Run 2: 4 new users → 37660 new rows

    Run 3: 6 new users → 56490 new rows

    Run 4: 25 new users → 235375 new rows

    Run 5: 50 new users → 470750 new rows

    Run 6: 100 new users → 941500 new rows

    Run 7: 200 new users → 1883000 new rows

    Run 8: 300 new users → 2824500 new rows

    Run 9: 350 new users → 3295250 new rows

    Run 10: 500 new users → 4707500 new rows

    Run 11: 1000 new users → 9415000 new rows

Genre: Adventure
  Positive books: 185
  Negative books: 9230
  Original users: 53424
  Original rows: 5976479
  Simulating synthetic users:

    Run 1: 2 new users → 18830 new rows

    Run 2: 4 new users → 37660 new rows

    Run 3: 6 new users → 56490 new rows

    Run 4: 25 new users → 235375 new rows

    Run 5: 50 new users → 470750 new rows

    Run 6: 100 new users → 941